# 05 — CLIP Embeddings
Génère les embeddings visuels (768 dims) de toutes les photos triées via `openai/clip-vit-large-patch14`.

Utilise Spark `mapInPandas` — exception justifiée aux règles no-UDF : le modèle PyTorch doit être chargé
en Python pur, il n'existe pas d'équivalent en Column expressions JVM.
Le modèle est chargé **une seule fois par partition**, pas par ligne.

**Input** : `data/processed/INSTAGRAM/CLIP_SORTING/`  
**Output** : `data/warehouse/photo_embeddings/`

In [1]:
import sys, os
from pathlib import Path

_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import PHOTO_EMBEDDINGS_DIR, PHOTO_SORTING_DIR, PROJECT_ROOT

PROJECT_ROOT = Path(PROJECT_ROOT)
PHOTOS_DIR   = Path(PHOTO_SORTING_DIR)
OUTPUT_DIR   = Path(PHOTO_EMBEDDINGS_DIR)

CLIP_MODEL   = "openai/clip-vit-large-patch14"  # ViT-L : 768 dims (vs 512 pour ViT-B)
IMAGE_EXTS   = {".jpg", ".jpeg", ".png", ".webp"}

# BATCH_SIZE : nombre d'images traitées par appel modèle.
# 32 = bon équilibre mémoire GPU/CPU vs overhead de chargement.
# Monter à 64 si GPU avec >8 Go VRAM ; descendre à 16 si OOM.
BATCH_SIZE   = 32

# N_PARTITIONS : nombre de workers Spark (= chargements du modèle CLIP).
# 4 partitions = 4 chargements de ~1.7 Go du modèle, un par cœur logique.
# Augmenter si le cluster a plus de workers ; garder bas en local pour éviter l'OOM.
N_PARTITIONS = 4

print(f"Racine projet : {PROJECT_ROOT}")
print(f"Photos dir    : {PHOTOS_DIR}")
print(f"Output dir    : {OUTPUT_DIR}")

Racine projet : /opt/spark
Photos dir    : /opt/spark/data/processed/INSTAGRAM/CLIP_SORTING
Output dir    : /opt/spark/data/warehouse/photo_embeddings


## 1. Collecter les chemins d'images

In [2]:
photo_paths = [
    {"path": str(p), "filename": p.name}
    for p in PHOTOS_DIR.rglob("*")
    if p.suffix.lower() in IMAGE_EXTS
]
print(f"Photos trouvées : {len(photo_paths)}")

Photos trouvées : 2420


## 2. Spark Session

In [3]:
from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

from config import build_spark_session
spark = build_spark_session("MyDigitalTwin-CLIP-Embeddings", delta=True, snappy=True)
spark.sparkContext.setLogLevel("WARN")

input_schema = StructType([
    StructField("path",     StringType(), nullable=False),
    StructField("filename", StringType(), nullable=False),
])

df = (
    spark.createDataFrame(photo_paths, schema=input_schema)
         .repartition(N_PARTITIONS)
)
print(f"DataFrame Spark : {df.count()} lignes, {N_PARTITIONS} partitions")

26/05/12 17:40:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


DataFrame Spark : 2420 lignes, 4 partitions


## 3. Fonction d'embedding (une fois par partition)

In [4]:
EMBED_SCHEMA = StructType([
    StructField("path",      StringType(),           nullable=False),
    StructField("filename",  StringType(),           nullable=False),
    StructField("embedding", ArrayType(FloatType()), nullable=True),
])

def embed_partition(iterator):
    import torch
    from PIL import Image
    from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor

    # CUDA si GPU disponible (inférence ~10x plus rapide qu'en CPU).
    # CPU en fallback : Docker sans GPU ou machine locale.
    # La détection est automatique — aucune config manuelle nécessaire.
    device    = "cuda" if torch.cuda.is_available() else "cpu"

    # Le modèle est chargé UNE FOIS par partition (pas par ligne).
    # Avec N_PARTITIONS=4, le modèle (~1.7 Go) est chargé 4 fois au total.
    # CLIPVisionModelWithProjection : retourne .image_embeds (tensor garanti),
    # contrairement à CLIPModel.get_image_features() qui retourne un objet
    # BaseModelOutputWithPooling selon la version de transformers.
    processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)
    model     = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL).to(device)
    model.eval()

    for pdf in iterator:
        paths      = pdf["path"].tolist()
        embeddings = []

        for i in range(0, len(paths), BATCH_SIZE):
            batch = paths[i : i + BATCH_SIZE]
            images, valid_idx = [], []

            for j, p in enumerate(batch):
                try:
                    images.append(Image.open(p).convert("RGB"))
                    valid_idx.append(j)
                except OSError:
                    pass

            result = [None] * len(batch)
            if images:
                inputs = processor(images=images, return_tensors="pt").to(device)
                with torch.no_grad():
                    feats = model(**inputs).image_embeds
                    # Normalisation L2 : embeddings sur l'hypersphère unité.
                    # Nécessaire pour que la distance cosine soit équivalente à la distance euclidienne.
                    feats = feats / feats.norm(dim=-1, keepdim=True)
                    feats = feats.cpu().float().numpy()
                for k, vi in enumerate(valid_idx):
                    result[vi] = feats[k].tolist()

            embeddings.extend(result)

        pdf = pdf.copy()
        pdf["embedding"] = embeddings
        yield pdf

## 4. Inférence CLIP + sauvegarde

In [5]:
from delta.tables import DeltaTable

df_embedded = df.mapInPandas(embed_partition, schema=EMBED_SCHEMA)
df_out = df_embedded.filter("embedding IS NOT NULL")
output_path = str(OUTPUT_DIR)

# ── Stratégie : MERGE INTO ────────────────────────────────────────────────────
# Type : table incrémentale — nouvelles photos ajoutées sans retraiter les existantes.
# Merge key : filename (stable et unique par photo).
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if DeltaTable.isDeltaTable(spark, output_path):
    DeltaTable.forPath(spark, output_path).alias("t") \
        .merge(df_out.alias("s"), "t.filename = s.filename") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    df_out.write.format("delta").mode("overwrite").save(output_path)

count = spark.read.format("delta").load(output_path).count()
print(f"Embeddings sauvegardés : {count} photos → {OUTPUT_DIR}")

26/05/12 18:03:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Embeddings sauvegardés : 2420 photos → /opt/spark/data/warehouse/photo_embeddings


In [7]:
spark.stop()